In [6]:
import os
import joblib
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

# --- SMOTE FOR REGRESSION (Outlier Booster) ---
def apply_smoter(X, y, threshold_quantile=0.95, samples_to_add=800):
    threshold = y.quantile(threshold_quantile)
    rare_indices = y[y >= threshold].index
    if len(rare_indices) < 5: return X, y
    X_synth, y_synth = [], []
    for _ in range(samples_to_add):
        idx1, idx2 = np.random.choice(rare_indices, 2)
        alpha = np.random.random()
        X_new = alpha * X.loc[idx1] + (1 - alpha) * X.loc[idx2]
        y_new = alpha * y.loc[idx1] + (1 - alpha) * y.loc[idx2]
        X_synth.append(X_new)
        y_synth.append(y_new)
    return pd.concat([X, pd.DataFrame(X_synth, columns=X.columns)], ignore_index=True), \
           pd.concat([y, pd.Series(y_synth)], ignore_index=True)

# --- BAYESIAN OBJECTIVE ---
def objective(trial, X, y, m_name):
    if m_name == "plsr":
        model = PLSRegression(n_components=trial.suggest_int("n_components", 2, 20))
    elif m_name == "rf":
        model = RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 1000),
            max_depth=trial.suggest_int("max_depth", 5, 50),
            n_jobs=-1, random_state=42
        )
    elif m_name == "xgb":
        model = XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 1500),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 15),
            n_jobs=-1, random_state=42
        )
    elif m_name == "lgbm": # <--- New search space for LGBM
        model = LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 1500),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 20, 256),
            feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
            random_state=42, n_jobs=-1, verbose=-1
        )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    return cross_val_score(pipeline, X, y, cv=5, scoring='r2', n_jobs=-1).mean()

# --- CONFIG & INITIALIZATION ---
DATA_PATH = r"data/spectral_features_triad_60k.csv"
SAVE_DIR = r"models_bayesian_v1"
os.makedirs(SAVE_DIR, exist_ok=True)
TARGETS = ['p1.pH.index', 'p1.EC.ds_m', 'p1.Clay.wt_pct', 'p1.Sand.wt_pct', 'p1.Silt.wt_pct', 'p2.N.wt_pct', 'p2.Zn.mg_kg', 'p2.OC.wt_pct', 'p3.Fe.mg_kg', 'p3.K.mg_kg', 'p3.P.mg_kg', 'p3.S.wt_pct', 'p4.BD.g_cm3', 'p4.CEC.cmolc_kg', 'p4.CF.wt_pct', 'p4.WR_10kPa.wt_pct', 'p4.WR_1500kPa.wt_pct', 'p4.WR_33kPa.wt_pct']

df = pd.read_csv(DATA_PATH)
spectral_cols = [c for c in df.columns if not c.startswith('p')]
additional_features = ["p1.pH.index", "p1.EC.ds_m"]

leaderboard_rows = []

In [7]:
for target in TARGETS:
    print(f"\n" + "="*40 + f"\nTARGET: {target}\n" + "="*40)
    mask = df[target].notna() & df["p1.pH.index"] & df["p1.EC.ds_m"]
    df_target = df[mask].copy()
    
    features = spectral_cols + additional_features
    X_train, X_test, y_train, y_test = train_test_split(df_target[features], df_target[target], test_size=0.2, random_state=42)
    X_train_boost, y_train_boost = apply_smoter(X_train, y_train)

    best_pipelines = {}
    cv_preds = {}

    for m_name in ["plsr", "rf", "xgb", "lgbm"]:
        print(f"  Optimizing {m_name.upper()}...")
        study = optuna.create_study(direction="maximize")
        study.optimize(lambda t: objective(t, X_train_boost, y_train_boost, m_name), n_trials=30)
        
        # Build best pipe using best params
        if m_name == "plsr": engine = PLSRegression(**study.best_params)
        elif m_name == "rf": engine = RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42)
        elif m_name == "xgb": engine = XGBRegressor(**study.best_params, n_jobs=-1, random_state=42)
        elif m_name == "lgbm": engine = LGBMRegressor(**study.best_params, n_jobs=-1, random_state=42, verbose=-1)
        
        best_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('model', engine)
        ])
        best_pipe.fit(X_train_boost, y_train_boost)
        best_pipelines[m_name] = best_pipe
        cv_preds[m_name] = cross_val_predict(best_pipe, X_train_boost, y_train_boost, cv=5)
        
        # Log the individual model results for our CSV
        leaderboard_rows.append({
            'Target': target,
            'Model': m_name,
            'Best_CV_R2': study.best_value,
            'Params': str(study.best_params)
        })

    # Train Stacker
    X_meta_train = pd.DataFrame(cv_preds)
    stacker = LinearRegression(positive=True)
    stacker.fit(X_meta_train, y_train_boost)

    # Final Holdout Evaluation
    test_base_preds = {m: pipe.predict(X_test).flatten() for m, pipe in best_pipelines.items()}
    X_meta_test = pd.DataFrame(test_base_preds)
    final_preds = stacker.predict(X_meta_test)
    holdout_r2 = r2_score(y_test, final_preds)
    
    # Add Stacker entry to leaderboard
    leaderboard_rows.append({
        'Target': target,
        'Model': 'STACKER_ENSEMBLE',
        'Best_CV_R2': np.nan,
        'Params': f"Weights: {list(stacker.coef_)}",
        'Holdout_R2': holdout_r2
    })

    # Save PKL
    save_pkg = {'target': target, 'pipelines': best_pipelines, 'stacker': stacker, 'features': features}
    joblib.dump(save_pkg, os.path.join(SAVE_DIR, f"{target}_bayesian_ensemble.pkl"))
    print(f"  ✅ {target} Done. Holdout R2: {holdout_r2:.4f}")

# --- FINAL CSV EXPORT ---
leaderboard_df = pd.DataFrame(leaderboard_rows)
leaderboard_df.to_csv(os.path.join(SAVE_DIR, "model_results_proper.csv"), index=False)
print(f"\nLeaderboard saved to {SAVE_DIR}/model_results_proper.csv")


TARGET: p1.pH.index


[I 2026-03-17 12:51:13,935] A new study created in memory with name: no-name-db1c8188-9fd5-4211-9277-a087c9a47c6e


  Optimizing PLSR...


[I 2026-03-17 12:51:16,515] Trial 0 finished with value: 0.9999898640563263 and parameters: {'n_components': 6}. Best is trial 0 with value: 0.9999898640563263.
[I 2026-03-17 12:51:19,062] Trial 1 finished with value: 1.0 and parameters: {'n_components': 20}. Best is trial 1 with value: 1.0.
[I 2026-03-17 12:51:21,589] Trial 2 finished with value: 0.999999999999212 and parameters: {'n_components': 15}. Best is trial 1 with value: 1.0.
[I 2026-03-17 12:51:23,984] Trial 3 finished with value: 0.9999999997418401 and parameters: {'n_components': 10}. Best is trial 1 with value: 1.0.
[I 2026-03-17 12:51:24,241] Trial 4 finished with value: 0.9999999960897032 and parameters: {'n_components': 9}. Best is trial 1 with value: 1.0.
[I 2026-03-17 12:51:24,501] Trial 5 finished with value: 0.9999999999981612 and parameters: {'n_components': 14}. Best is trial 1 with value: 1.0.
[I 2026-03-17 12:51:24,757] Trial 6 finished with value: 0.9999999999618321 and parameters: {'n_components': 11}. Best is

  Optimizing RF...


[I 2026-03-17 12:51:57,275] Trial 0 finished with value: 0.9999637880329704 and parameters: {'n_estimators': 236, 'max_depth': 35}. Best is trial 0 with value: 0.9999637880329704.
[I 2026-03-17 12:52:37,192] Trial 1 finished with value: 0.9991507764158882 and parameters: {'n_estimators': 764, 'max_depth': 5}. Best is trial 0 with value: 0.9999637880329704.
[I 2026-03-17 12:53:26,766] Trial 2 finished with value: 0.9998106843629806 and parameters: {'n_estimators': 710, 'max_depth': 6}. Best is trial 0 with value: 0.9999637880329704.
[I 2026-03-17 12:55:08,328] Trial 3 finished with value: 0.9999620215236542 and parameters: {'n_estimators': 890, 'max_depth': 48}. Best is trial 0 with value: 0.9999637880329704.
[I 2026-03-17 12:55:47,501] Trial 4 finished with value: 0.9999637244419268 and parameters: {'n_estimators': 352, 'max_depth': 48}. Best is trial 0 with value: 0.9999637880329704.
[I 2026-03-17 12:56:53,501] Trial 5 finished with value: 0.9999626938691817 and parameters: {'n_estima

  Optimizing XGB...


[I 2026-03-17 13:11:39,530] Trial 0 finished with value: 0.9995856438056918 and parameters: {'n_estimators': 751, 'learning_rate': 0.09949204121359377, 'max_depth': 9}. Best is trial 0 with value: 0.9995856438056918.
[I 2026-03-17 13:11:41,880] Trial 1 finished with value: 0.42099721679944474 and parameters: {'n_estimators': 223, 'learning_rate': 0.0012848834482933678, 'max_depth': 7}. Best is trial 0 with value: 0.9995856438056918.
[I 2026-03-17 13:11:46,231] Trial 2 finished with value: 0.9995855121330137 and parameters: {'n_estimators': 501, 'learning_rate': 0.14365530179820324, 'max_depth': 10}. Best is trial 0 with value: 0.9995856438056918.
[I 2026-03-17 13:11:48,134] Trial 3 finished with value: 0.9997035873923107 and parameters: {'n_estimators': 1271, 'learning_rate': 0.04602684451303182, 'max_depth': 3}. Best is trial 3 with value: 0.9997035873923107.
[W 2026-03-17 13:11:57,915] Trial 4 failed with parameters: {'n_estimators': 1073, 'learning_rate': 0.026206481941932366, 'max_

KeyboardInterrupt: 